In [ ]:
import torch
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

login(token=HF_TOKEN)

print("PyTorch version:", torch.__version__)
print("CUDA Available", torch.cuda.is_available())
if torch.cuda.is_available():
  print("Device name", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 2.8 MB/s eta 0:00:00
PyTorch version: 2.11.0+cu128
CUDA Available True
Device name Tesla T4


In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MODEL_CKPT = "Davlan/afro-xlmr-base"
FOLDER_PATH = "../data/processed"

train_df = pd.read_csv(os.path.join(FOLDER_PATH, "train.csv"))
val_df = pd.read_csv(os.path.join(FOLDER_PATH, "val.csv"))
test_df = pd.read_csv(os.path.join(FOLDER_PATH, "test.csv"))

pos_df = train_df[train_df["label"] == 1]

train_df = pd.concat([train_df, pos_df, pos_df], ignore_index=True)
train_df = train_df.sample(frac=1.0, random_state=45).reset_index(drop=True)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT, token=HF_TOKEN)

print("Successfully fetched tokenizer")

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Successfully fetched tokenizer


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import numpy as np
import torch.nn as nn

class_weights = compute_class_weight(
    class_weight="balanced", classes=np.array([0, 1, 2]), y=train_df["label"]
)
weight_tensor = torch.tensor(class_weights, dtype=torch.float)
print(f"Computed Class Weights: {weight_tensor}")


class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            weight_tensor = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=weight_tensor)
        else:
            loss_fct = nn.CrossEntropyLoss()

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def tokenizer_batch(example):
    return tokenizer(example["tweet"], truncation=True, max_length=128)


tokenized_train = train_dataset.map(tokenizer_batch, batched=True)
tokenized_val = val_dataset.map(tokenizer_batch, batched=True)
tokenized_test = test_dataset.map(tokenizer_batch, batched=True)

id2label = {0: "Positive", 1: "Neutral", 2: "Negative"}
label2id = {"Positive": 0, "Neutral": 1, "Negative": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT, num_labels=3, id2label=id2label, label2id=label2id, token=HF_TOKEN
)

Computed Class Weights: tensor([0.9934, 5.6248, 0.5508])


Map:   0%|          | 0/6041 [00:00<?, ? examples/s]

Map:   0%|          | 0/1295 [00:00<?, ? examples/s]

Map:   0%|          | 0/1295 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import EarlyStoppingCallback


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro"
    )
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "macro_f1": f1, "precision": precision, "recall": recall}


num_train_epochs = 6
per_device_train_batch_size = 16
warmup_ratio = 0.1
total_steps = int(len(tokenized_train) / per_device_train_batch_size) * num_train_epochs
warmup_steps = int(total_steps * warmup_ratio)

training_args = TrainingArguments(
    output_dir="./afro-xlmr-weighted",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=16,
    num_train_epochs=num_train_epochs,
    weight_decay=0.1,
    label_smoothing_factor=0.15,
    warmup_steps=warmup_steps,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    seed=45,
    fp16=True,
)

trainer = WeightedTrainer(
    class_weights=weight_tensor,
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

trainer.save_model("./afro-xlmr-weighted")
tokenizer.save_pretrained("./afro-xlmr-weighted")

print("Training complete and best model saved to ./afro-xlmr-weighted")

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision,Recall
1,1.091147,1.042370,0.559073,0.460704,0.480324,0.481511
2,0.969110,0.964098,0.608494,0.509533,0.504968,0.557587
3,0.856104,0.957321,0.658687,0.541977,0.533675,0.556756
4,0.795530,0.990250,0.675676,0.555317,0.558572,0.556115
5,0.766894,1.024196,0.673359,0.556090,0.549895,0.566207
6,0.709557,1.043843,0.684942,0.567554,0.565759,0.570550


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and best model saved to ./afro-xlmr-weighted
